# SQL Basics

**Estimated time:** ~10 hours total (about 4 hours reading and running this guide + ~6 hours on
`sql-basics-exercises.ipynb`).

SQL is how you ask a database a question. It has been the answer to that problem since the 1970s, it
outlived every language that tried to replace it, and it is still the first thing on almost every data
job description. The good news is that the useful part is small: a dozen keywords cover most of the
queries written in a working day.

## Who This Is For

You have never written SQL, or you have copied a few queries without really knowing what they do. You
know what a spreadsheet is. Python helps here — this notebook runs its queries from Python — but you are
not learning Python, you are learning SQL. Every query in this notebook is plain SQL that would work the
same way pasted into a database client at work.

## What You Will Learn

- What a table, a row, a column, a primary key and a foreign key actually are
- `SELECT`, `WHERE`, `ORDER BY`, `LIMIT` — pulling the rows you want out of one table
- `NULL`, and why it breaks the comparisons you expect to work
- Aggregates, `GROUP BY` and `HAVING` — turning many rows into a summary
- `CASE WHEN`, `COALESCE` — making decisions inside a query
- `INNER JOIN` and `LEFT JOIN` — reading two tables at once, and what the difference costs you
- The order a query really runs in, which explains most beginner error messages
- `CREATE`, `INSERT`, `UPDATE`, `DELETE` — changing data, not just reading it
- When to do the work in SQL and when to pull the rows into pandas

## How to Use This Guide

Run every cell yourself with **Shift + Enter**, in order. The first code cell builds a small shop database in
memory from the bundled CSV files in `../assets/sql/`, so nothing needs downloading, nothing is left on disk,
and you can restart the kernel at any time and start again.

Read the query before you run it and guess the shape of the answer — how many rows, which columns. Then run
it and see whether you were right. Guessing first is what turns reading into learning.

Change the queries. Break them. The error messages are part of the material, and a database is very hard to
damage when the whole thing is rebuilt by the first cell.

The engine here is **SQLite**, which ships inside Python — no server, no install, no password. Everything you
learn transfers to PostgreSQL, MySQL, SQL Server and BigQuery; section 24 lists the handful of places where
the spelling changes.

**Practice:** `sql-basics-exercises.ipynb` follows this guide section by section.

## 1. What a Database Is — And How to Run These Queries

A database is a set of **tables**. A table is a grid: **columns** have names and types, **rows** are the
records. That is the whole mental model, and it is not far from a spreadsheet. What a database adds is that
it enforces the shape (a price column holds numbers, and nothing else), it can link tables to each other,
and it can answer questions over millions of rows without you writing a loop.

SQL is the language for asking. You do not tell the database *how* to find the rows — no loops, no indexes
by hand. You describe *what* you want and the engine works out how to get it.

The cell below builds our practice database. It reads `schema.sql` (which creates seven empty tables) and
seven CSV files from `../assets/sql/`, and gives you two small helpers you will use all the way through:
`q(...)` runs a query and hands back the answer as a pandas table, and `run(...)` runs a statement that
changes something.

In [8]:
import sqlite3
import pandas as pd

SQL = "../assets/sql"          # the bundled schema and CSV files
TABLES = ["categories", "customers", "employees", "products",
          "orders", "order_items", "payments"]

con = sqlite3.connect(":memory:")         # the database lives in RAM -- nothing to clean up

with open(f"{SQL}/schema.sql") as f:
    con.executescript(f.read())           # creates the seven empty tables

con.execute("PRAGMA foreign_keys = ON")   # from here on, SQLite enforces the foreign keys

for table in TABLES:
    pd.read_csv(f"{SQL}/{table}.csv").to_sql(table, con, if_exists="append", index=False)
con.commit()


def q(sql):
    """Run a SELECT and hand the result back as a pandas DataFrame."""
    return pd.read_sql_query(sql, con)


def run(sql):
    """Run statements that change data or structure: CREATE, INSERT, UPDATE, DELETE."""
    con.executescript(sql)
    con.commit()


pd.set_option("display.width", 110)
pd.set_option("display.max_rows", 25)

for table in TABLES:
    print(f"{table:12s} {q(f'SELECT COUNT(*) AS n FROM {table}')['n'][0]:>4} rows")

FileNotFoundError: [Errno 2] No such file or directory: '../assets/sql/schema.sql'

**Step by step:**

1. `sqlite3` is part of Python's standard library, so there is nothing to install. `sqlite3.connect(":memory:")`
   makes a database that lives in RAM: it is built fresh every time you run this cell and it leaves no file
   behind. Swap in a filename and the same code writes a real database on disk.
2. `con.executescript(...)` runs the whole of `schema.sql`, which drops any existing tables and creates them
   again. That is what makes this cell safe to re-run.
3. `PRAGMA foreign_keys = ON` tells SQLite to actually enforce the links between tables. SQLite leaves this off
   by default, which surprises people; other databases enforce them always. It comes *after* the schema script,
   because dropping a table other tables point at is exactly the thing the setting would block.
4. The loop reads each CSV with pandas and pushes it into the matching table with `to_sql(..., if_exists="append")`.
   The tables already exist with proper types, so `append` fills them rather than replacing them.
5. `q(sql)` wraps `pd.read_sql_query`, so the result of a `SELECT` comes back as a DataFrame that Jupyter
   prints as a neat grid. `run(sql)` is for statements with no result to show.
6. The final loop prints a row count per table, so you can see at a glance that the load worked.

## 2. The Shop Schema — Tables, Rows, Columns and Keys

Our database is a small online electronics shop. Seven tables:

| Table | Rows | What it holds |
| --- | --- | --- |
| `categories` | 8 | Laptops, Phones, Audio, Accessories, Monitors, Storage, Cameras, Wearables |
| `customers` | 60 | who buys — name, email, city, state, signup date |
| `employees` | 15 | staff, with `manager_id` pointing back at the same table |
| `products` | 40 | list `price`, our `cost`, and `stock` |
| `orders` | 300 | one row per order — customer, date, status, channel |
| `order_items` | 673 | the lines inside an order — quantity, unit price, discount |
| `payments` | 248 | one row per paid order |

Two ideas hold it together:

- A **primary key** is the column that identifies a row uniquely. `customers.customer_id` is one. No two
  customers share it, and it is never empty.
- A **foreign key** is a column that points at another table's primary key. `orders.customer_id` is one: it
  says which customer placed the order. That single link is why you never repeat the customer's name and city
  on every order.

```
categories --< products --< order_items >-- orders >-- customers
                                              |
                                              +--< payments
                                              +-- employees (manager_id --> employees)
```

Read `--<` as "one row here, many rows there". One customer has many orders; one order has many items.

The data is deliberately imperfect — five customers never filled in their city, nine never ordered anything,
fifty-two orders were never paid for. That is what real data looks like, and half of this guide is about
handling it.

In [3]:
print(q("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")["name"].tolist())

q("PRAGMA table_info(customers)")

NameError: name 'q' is not defined

**Step by step:**

1. `sqlite_master` is a table SQLite keeps about itself — every table, index and view it holds. Querying it
   is how you look around a database you have never seen before.
2. `WHERE type = 'table'` filters out indexes and views. Note the **single quotes**: in SQL, single quotes are
   for text values. Double quotes mean "this is a column or table name", which is a trap worth remembering.
3. `["name"].tolist()` is pandas, not SQL — it turns the one-column result into a plain Python list so the
   print is short.
4. `PRAGMA table_info(customers)` is SQLite's way of describing one table: the column names, their declared
   types, whether they can be empty (`notnull`), and which column is the primary key (`pk`).
5. Because it is the last expression in the cell, Jupyter displays it as a table. Every other database has its
   own spelling for this — `\d customers` in PostgreSQL, `DESCRIBE customers` in MySQL.

## 3. SELECT — Asking for Columns

Every query that reads data starts with `SELECT`. You name the columns you want, then `FROM` names the table
they live in.

`SELECT *` means "every column". It is fine while you are exploring and a bad habit in code you keep: it
breaks the moment somebody adds a column, it drags data you do not need across the network, and it hides
what the query actually depends on. Name your columns.

`AS` renames a column in the output. The stored column keeps its name; only the result changes. This matters
more than it sounds — the moment you compute something, the output column has no name of its own, and a
result with a column called `ROUND((price - cost) * 100.0 / price, 1)` is nobody's friend.

In [ ]:
q("""
SELECT name        AS customer,
       city,
       signup_date AS joined
FROM customers
ORDER BY customer_id
LIMIT 5
""")

**Step by step:**

1. The three column names after `SELECT` are separated by commas. Line breaks and extra spaces mean nothing to
   SQL, so lay a query out to be read.
2. `name AS customer` renames `name` to `customer` in the result. The `AS` is optional (`name customer` works)
   but leaving it out reads like a typo, so keep it.
3. `city` is asked for without a rename and keeps its own name.
4. `FROM customers` says which table. Without it there are no columns to select.
5. `LIMIT 5` cuts the answer to five rows. With `ORDER BY customer_id` in front of it, those are the first five
   by id — `LIMIT` without `ORDER BY` gives you five arbitrary rows, which is rarely what you meant.

## 4. WHERE — Keeping Only the Rows You Want

`WHERE` filters rows. The database looks at each row, evaluates your condition, and keeps the rows where it
comes out true.

The comparison operators are the ones you would guess — `=`, `<>` (not equal, and `!=` also works), `<`, `<=`,
`>`, `>=`. The one that catches people out is equality: SQL uses a single `=` for comparison, not `==`.

Text goes in single quotes: `WHERE city = 'Chennai'`. Numbers do not: `WHERE price > 50000`. Comparisons on
text are case-sensitive in most databases — SQLite compares `'chennai'` and `'Chennai'` as different values.

In [ ]:
q("""
SELECT name, price, stock
FROM products
WHERE price > 50000
ORDER BY price DESC
""")

**Step by step:**

1. `FROM products` gives the query 40 rows to work with.
2. `WHERE price > 50000` tests each of them. Rows where the answer is true survive; the rest are dropped before
   anything else happens.
3. Nothing is quoted around `50000` because it is a number. Quoting it would compare a number against text,
   and SQLite would quietly try to convert — other databases would throw an error.
4. `ORDER BY price DESC` sorts what is left, most expensive first.
5. The result has only the three columns named in `SELECT`, even though `WHERE` was allowed to look at all of
   them. Filtering happens before the output is built, so `WHERE` can use columns you never display.

## 5. AND, OR, NOT — Combining Conditions

`AND` keeps rows where both sides are true. `OR` keeps rows where either side is. `NOT` flips a condition.

`AND` binds tighter than `OR`, exactly like `*` binds tighter than `+`. So

```sql
WHERE category_id = 1 OR category_id = 2 AND price < 30000
```

means "category 1, **or** (category 2 under 30000)" — almost certainly not what was meant. Put brackets around
the `OR` and stop thinking about precedence. Every experienced person writes the brackets; nobody is showing
off by leaving them out.

In [ ]:
q("""
SELECT name, category_id, price, stock
FROM products
WHERE (category_id = 1 OR category_id = 2)
  AND price < 30000
  AND stock > 50
ORDER BY price
""")

**Step by step:**

1. The bracketed `OR` runs first: keep laptops (`category_id = 1`) and phones (`category_id = 2`).
2. `AND price < 30000` narrows that to the affordable ones. Every laptop is above 30000, so the laptops all
   drop out here.
3. `AND stock > 50` narrows again to what we actually have on the shelf.
4. All three conditions must hold for a row to survive. Chaining `AND`s down the page, one per line, is the
   normal way to write a filter that has grown past one line.
5. Try removing the brackets and re-running. The answer changes, and the query still runs — SQL will not warn
   you that you meant something else.

## 6. BETWEEN, IN and LIKE — Shorter Ways to Say Common Things

Three pieces of shorthand you will use constantly:

- `x BETWEEN a AND b` is `x >= a AND x <= b`. **Both ends are included.** With dates that trips people up:
  `BETWEEN '2024-01-01' AND '2024-01-31'` includes the 31st, which is usually right, but
  `BETWEEN '2024-01-01' AND '2024-02-01'` sneaks in one day of February.
- `x IN (a, b, c)` is `x = a OR x = b OR x = c`, without the brackets-and-precedence problem.
- `x LIKE 'pattern'` matches text. `%` stands for any run of characters (including none) and `_` for exactly
  one. `LIKE 'Vault%'` finds things starting with Vault; `LIKE '%SSD%'` finds it anywhere in the name.

`NOT IN`, `NOT BETWEEN` and `NOT LIKE` all exist and all behave as you would expect — with one exception around
`NULL` that we come to in the next section, and again, less forgivingly, in the intermediate level.

In [ ]:
cameras_and_watches = q("SELECT name, category_id FROM products WHERE category_id IN (7, 8)")
mid_priced = q("SELECT name, price FROM products WHERE price BETWEEN 20000 AND 30000")

print(len(cameras_and_watches), "cameras and wearables")
print(len(mid_priced), "products priced 20000 to 30000 inclusive")

q("""
SELECT name, price
FROM products
WHERE name LIKE '%Laptop%'
   OR name LIKE 'Vault%'
ORDER BY name
""")

**Step by step:**

1. `category_id IN (7, 8)` picks cameras and wearables in one go. With a long list this is far easier to read
   than a chain of `OR`s.
2. `price BETWEEN 20000 AND 30000` keeps products at exactly 20000 and exactly 30000 too — the ends are in.
3. Both of those results are stored in Python variables rather than displayed, so the `print` lines can report
   just their sizes. `len(df)` is the row count.
4. `name LIKE '%Laptop%'` matches anywhere in the string: `Aster 14 Laptop` and `Aster 15 Pro Laptop` both hit.
   `LIKE 'Vault%'` is anchored at the start, so it matches the three Vault drives and nothing else.
5. A leading `%` means the database cannot use an index to jump straight to the matching rows — it has to read
   every one. On 40 products that is free. On 40 million it is the difference between a query and a coffee
   break, which is the subject of the advanced level.

## 7. NULL — The Value That Is Not a Value

`NULL` means *unknown*. It is not zero, not an empty string, not `False`. Five of our customers never gave a
city, so their `city` is `NULL`.

The rule that follows from "unknown" is the one that catches everybody: **any comparison with `NULL` is
neither true nor false — it is unknown**, and `WHERE` only keeps rows where the condition came out true. So
`WHERE city = NULL` matches nothing. Not even the rows that are `NULL`. There is no error and no warning; you
just get an empty result and a wrong conclusion.

The fix is a different operator: `IS NULL` and `IS NOT NULL`.

The same logic bites in a place people do not expect. `WHERE city <> 'Chennai'` drops the `NULL` rows, because
"is an unknown city different from Chennai?" is itself unknown. If you want them, you have to say so:
`WHERE city <> 'Chennai' OR city IS NULL`.

In [ ]:
wrong = q("SELECT COUNT(*) AS n FROM customers WHERE city = NULL")
right = q("SELECT COUNT(*) AS n FROM customers WHERE city IS NULL")
print("WHERE city =  NULL  ->", wrong["n"][0], "rows")
print("WHERE city IS NULL  ->", right["n"][0], "rows")

not_chennai      = q("SELECT COUNT(*) AS n FROM customers WHERE city <> 'Chennai'")
not_chennai_full = q("SELECT COUNT(*) AS n FROM customers WHERE city <> 'Chennai' OR city IS NULL")
print("city <> 'Chennai'                     ->", not_chennai["n"][0], "rows")
print("city <> 'Chennai' OR city IS NULL     ->", not_chennai_full["n"][0], "rows")

q("SELECT customer_id, name, city, state FROM customers WHERE city IS NULL ORDER BY customer_id")

**Step by step:**

1. The first pair is the whole lesson. `= NULL` returns 0 rows; `IS NULL` returns 5. Same intent, silently
   different answers.
2. `COUNT(*)` counts the surviving rows. `["n"][0]` is pandas pulling the single number out of the
   one-cell result.
3. The second pair shows the leak. There are 60 customers and 12 are in Chennai, so you might expect 48 rows
   from `city <> 'Chennai'` — you get 43, because the five unknown cities silently vanish.
4. Adding `OR city IS NULL` brings them back and the numbers add up again.
5. The last query lists the five rows. In the displayed result pandas shows SQL's `NULL` as `None`, which is
   worth knowing before you go looking for the string `"NULL"`.

## 8. ORDER BY — Putting Rows in an Order You Choose

Rows in a table have **no order**. None. A database is free to hand them back in whatever order is convenient,
and that order can change when the data changes. If you want an order, you ask for one.

`ORDER BY column` sorts ascending (`ASC`, the default). `DESC` reverses it. Several columns separated by commas
sort by the first, then break ties with the second, and so on.

Ties matter more than they look. If you sort 300 orders by date only and take the top 10, the rows *inside* a
date can come back in any order — so the same query can give different answers on different days. Adding a
unique tiebreaker (`ORDER BY order_date DESC, order_id DESC`) makes the result reproducible, which is why the
exercises in this course always ask for one.

In [ ]:
q("""
SELECT name, category_id, price
FROM products
ORDER BY category_id ASC, price DESC
LIMIT 10
""")

**Step by step:**

1. `ORDER BY category_id ASC` groups the output visually by category, lowest id first. `ASC` is the default and
   could be left out; writing it makes the intent obvious next to the `DESC` on the following line.
2. `, price DESC` breaks ties: inside each category, dearest first. The comma-separated list is a priority
   order, not two separate sorts.
3. `LIMIT 10` takes the first ten rows *after* sorting. Sorting always happens before the limit, which is why
   "top 10 by price" is just `ORDER BY price DESC LIMIT 10`.
4. Because `category_id` values 1 and 2 fill the ten rows, you see the laptops and the start of the phones —
   swap the two `ORDER BY` terms and you get a completely different ten rows.
5. You can sort by a column you did not select. `ORDER BY stock DESC` works here even though `stock` is not in
   the output.

## 9. LIMIT and OFFSET — Looking at a Slice

`LIMIT n` returns at most `n` rows. `OFFSET m` skips the first `m` before it starts counting. Together they
give you pages: `LIMIT 5 OFFSET 0` is page one, `LIMIT 5 OFFSET 5` is page two.

Two habits worth forming now. First, `LIMIT` without `ORDER BY` is meaningless — you are asking for "any five
rows" and the database is allowed to change its mind. Second, while you are exploring an unfamiliar table, put
`LIMIT 10` on everything. It costs nothing and it saves you from accidentally pulling ten million rows into
your notebook.

In [ ]:
page1 = q("SELECT name, price FROM products ORDER BY price DESC, name LIMIT 5")
page2 = q("SELECT name, price FROM products ORDER BY price DESC, name LIMIT 5 OFFSET 5")

print("PAGE 1")
print(page1.to_string(index=False))
print("\nPAGE 2")
print(page2.to_string(index=False))

**Step by step:**

1. Both queries sort the same way — `price DESC` with `name` as the tiebreaker — so the two pages fit together
   with nothing missing and nothing repeated.
2. Page one takes rows 1 to 5. `OFFSET` is left out, which is the same as `OFFSET 0`.
3. Page two skips five rows and takes the next five, so it is rows 6 to 10 of the same sorted list.
4. `to_string(index=False)` is pandas formatting, not SQL — it prints the frame without the row numbers that
   would otherwise sit on the left.
5. `OFFSET` on a big table is slower than it looks: to skip a million rows the database still has to walk past
   them. Real pagination over large tables usually filters on the last id seen instead
   (`WHERE product_id > 40 ORDER BY product_id LIMIT 5`).

## 10. DISTINCT — One Row per Distinct Value

`SELECT DISTINCT` throws away duplicate rows from the result. With one column it answers "which values appear
at all?"; with several it de-duplicates the whole combination, not each column separately.

`COUNT(DISTINCT column)` counts how many different values there are.

Be a little suspicious of `DISTINCT`. When somebody adds it to a query because rows were showing up twice, it
is usually hiding a join that is producing too many rows rather than fixing it — a habit the intermediate
level takes apart properly. Used deliberately, to answer "what values exist?", it is exactly right.

In [ ]:
print(q("SELECT COUNT(*) AS n FROM customers")["n"][0], "customer rows")
print(q("SELECT COUNT(DISTINCT city) AS n FROM customers")["n"][0], "different cities")
print(q("SELECT COUNT(DISTINCT email) AS n FROM customers")["n"][0], "different email addresses")

q("SELECT DISTINCT city, state FROM customers ORDER BY state, city")

**Step by step:**

1. 60 customer rows, but only 58 different email addresses — two people signed up twice. Comparing a row count
   with a distinct count is the fastest duplicate check there is.
2. `COUNT(DISTINCT city)` returns 11, not 12. `COUNT` of a column **ignores `NULL`s**, so the five customers
   with no city are not counted as a twelfth kind of city.
3. `SELECT DISTINCT city, state` de-duplicates the pair. Mumbai and Pune both map to Maharashtra and stay as
   two separate rows, because the pairs differ.
4. The `NULL` city does come through as a row here — `DISTINCT` treats `NULL` as a value like any other, which
   is the opposite of what `COUNT` just did. The inconsistency is real; there is no trick to it beyond
   remembering which is which.
5. `ORDER BY state, city` sorts by state first so the same-state cities sit together.

## 11. Computed Columns — Doing Arithmetic in the Query

Anything after `SELECT` can be an expression, not just a column name. `price - cost` is a column that does not
exist in the table and appears in the result anyway.

Two things to watch. First, **integer division truncates**: `7 / 2` is `3`, not `3.5`, because both sides are
integers. Multiplying by `1.0` (or `100.0` instead of `100`) forces the calculation into decimals. Second, any
arithmetic involving `NULL` produces `NULL` — one unknown poisons the whole expression, which is why
`COALESCE` in section 18 exists.

`ROUND(x, n)` rounds to `n` decimal places, which keeps percentages readable.

In [ ]:
q("""
SELECT name,
       price,
       cost,
       price - cost                            AS margin,
       ROUND((price - cost) * 100.0 / price, 1) AS margin_pct,
       price * stock                           AS stock_value
FROM products
ORDER BY margin_pct DESC, name
LIMIT 8
""")

**Step by step:**

1. `price - cost` is evaluated for every row and shown as `margin`. Without the `AS` the column would be named
   after the expression itself.
2. `(price - cost) * 100.0 / price` is the margin as a percentage. The `100.0` is doing real work: with plain
   `100`, every value in this column would be a truncated integer.
3. `ROUND(..., 1)` keeps one decimal place. `ROUND(x)` with no second argument rounds to a whole number.
4. `price * stock` values the shelf. Computed columns can be as involved as you like — they are just
   arithmetic applied row by row.
5. `ORDER BY margin_pct DESC` sorts by the alias. That works in SQLite and PostgreSQL because `ORDER BY` runs
   after `SELECT` has built the output; the same trick in `WHERE` is not portable, which section 21 explains.

## 12. Text Functions — Cleaning Up Strings

The everyday string functions, all of which exist under these names in SQLite and, with small differences,
everywhere else:

| Function | What it does |
| --- | --- |
| `UPPER(s)` / `LOWER(s)` | change case — the usual way to compare text without caring about case |
| `LENGTH(s)` | number of characters |
| `TRIM(s)` | strip leading and trailing spaces (`LTRIM`, `RTRIM` for one side) |
| `SUBSTR(s, start, len)` | a slice. **Positions start at 1**, not 0 |
| `INSTR(s, sub)` | position of `sub` in `s`, or 0 if it is not there |
| `REPLACE(s, from, to)` | replace every occurrence |
| `a \|\| b` | glue two strings together |

The one that surprises Python programmers is `SUBSTR` counting from 1. SQL is one-based nearly everywhere it
counts positions.

In [ ]:
q("""
SELECT name,
       UPPER(city)                             AS city_upper,
       LENGTH(name)                            AS name_length,
       SUBSTR(email, 1, INSTR(email, '@') - 1) AS username,
       SUBSTR(email, INSTR(email, '@') + 1)    AS domain,
       name || ' <' || email || '>'            AS display_name
FROM customers
ORDER BY customer_id
LIMIT 5
""")

**Step by step:**

1. `UPPER(city)` shouts the city. On the rows where `city` is `NULL` the result is `NULL` too — string
   functions pass unknowns straight through.
2. `LENGTH(name)` counts characters including the space in the middle.
3. `INSTR(email, '@')` finds where the `@` sits. `SUBSTR(email, 1, that - 1)` therefore takes everything before
   it: the username. The `- 1` is there because `INSTR` returns the position *of* the `@`.
4. `SUBSTR(email, INSTR(email, '@') + 1)` with no length argument runs to the end of the string, giving the
   domain.
5. `||` is SQL's string concatenation. Not `+` — in most databases `+` on text is either an error or a silent
   attempt to add numbers. MySQL is the odd one out and wants `CONCAT(a, b)`.

## 13. Dates — Storing Them as Text, and Doing Maths on Them

SQLite has no date type. Dates live in `TEXT` columns as `'YYYY-MM-DD'`, and that format is chosen on purpose:
sorted alphabetically, `'2023-12-31'` comes before `'2024-01-01'`, so ordinary text comparison does the right
thing. `WHERE order_date >= '2024-01-01'` works exactly as you would hope.

Two functions do the rest:

- `strftime(format, date)` pulls pieces out — `'%Y'` year, `'%m'` month, `'%d'` day, `'%Y-%m'` the year-month
  bucket you will use in every monthly report you ever write.
- `date(d, modifier)` does arithmetic: `date('2024-03-01', '-1 day')`, `date(order_date, '+7 days')`,
  `date(order_date, 'start of month')`.

`julianday(d)` turns a date into a number so you can subtract two of them and get days.

Other databases have real date types and their own function names — `EXTRACT` and `DATE_TRUNC` in PostgreSQL,
`DATE_FORMAT` in MySQL. The ideas are identical; only the spelling moves.

In [ ]:
q("""
SELECT order_id,
       order_date,
       strftime('%Y',    order_date)                    AS year,
       strftime('%Y-%m', order_date)                    AS month,
       date(order_date, 'start of month')               AS month_start,
       date(order_date, '+7 days')                      AS delivery_due,
       CAST(julianday('2025-01-01') - julianday(order_date) AS INTEGER) AS days_before_2025
FROM orders
WHERE order_date >= '2024-12-01'
ORDER BY order_date DESC, order_id DESC
LIMIT 6
""")

**Step by step:**

1. `WHERE order_date >= '2024-12-01'` is a plain text comparison, and it is correct because of the
   `YYYY-MM-DD` format. Store dates as `'01/12/2024'` and this stops working — that is why the format matters.
2. `strftime('%Y-%m', order_date)` produces `'2024-12'`. Grouping by that string is how you get a monthly
   report, and it is the single most-used date expression in analytics.
3. `date(order_date, 'start of month')` gives a real date instead of a label — `'2024-12-01'`. Use this when
   the result has to sort or join against other dates.
4. `date(order_date, '+7 days')` adds a week. The modifiers chain: `date(d, 'start of month', '+1 month', '-1 day')`
   is the last day of the month.
5. `julianday` counts days since a fixed point thousands of years ago, so the difference of two of them is a
   number of days. `CAST(... AS INTEGER)` chops the fractional part off for display.

## 14. Aggregate Functions — Collapsing Many Rows into One

An **aggregate** takes many rows and returns one value: `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`. With no `GROUP BY`
the whole table is one group, so the answer is a single row.

The distinction that matters is `COUNT(*)` versus `COUNT(column)`:

- `COUNT(*)` counts **rows**.
- `COUNT(column)` counts rows where that column **is not NULL**.

`SUM`, `AVG`, `MIN` and `MAX` all skip `NULL`s too. That is usually what you want, but `AVG` deserves a second
look: if a third of your rows are `NULL`, `AVG` is the average of the other two thirds, not of everything with
the missing ones treated as zero. Nobody is told when this happens.

In [ ]:
print(q("""
SELECT COUNT(*)           AS all_orders,
       COUNT(employee_id) AS handled_by_a_rep
FROM orders
""").to_string(index=False))

q("""
SELECT COUNT(*)             AS products,
       ROUND(AVG(price), 2) AS avg_price,
       MIN(price)           AS cheapest,
       MAX(price)           AS dearest,
       SUM(price * stock)   AS stock_value
FROM products
""")

**Step by step:**

1. `COUNT(*)` says 300 orders. `COUNT(employee_id)` says 93 — the web and app orders have no sales rep, so
   their `employee_id` is `NULL` and they are not counted.
2. That gap is a feature, not a nuisance: `COUNT(some_nullable_column)` is the quickest way to ask "how many
   rows have this filled in?".
3. The second query aggregates five different ways over the same 40 rows in one pass. Each function gets its
   own output column.
4. `ROUND(AVG(price), 2)` — aggregates nest inside ordinary functions like anything else.
5. `SUM(price * stock)` multiplies first, row by row, and then adds the results. It is not `SUM(price) *
   SUM(stock)`, which would be a meaningless number.

## 15. GROUP BY — One Row per Group

`GROUP BY` splits the rows into groups and runs the aggregates once per group. `GROUP BY channel` gives you one
row per channel.

The rule to internalise: **every column in your `SELECT` must either be in the `GROUP BY` or be inside an
aggregate.** Anything else has no single value for the group, so the question is meaningless. PostgreSQL
rejects it outright. SQLite and older MySQL will run it and silently pick a value from an arbitrary row in the
group, which is worse than an error because you get a plausible wrong answer.

`NULL` forms its own group. All five city-less customers land together in one `NULL` row.

In [ ]:
print(q("""
SELECT channel,
       COUNT(*) AS orders
FROM orders
GROUP BY channel
ORDER BY orders DESC
""").to_string(index=False))

q("""
SELECT status,
       COUNT(*)                     AS orders,
       COUNT(DISTINCT customer_id)  AS customers,
       MIN(order_date)              AS first_seen,
       MAX(order_date)              AS last_seen
FROM orders
GROUP BY status
ORDER BY orders DESC
""")

**Step by step:**

1. `GROUP BY channel` finds the four distinct channels and makes a group of each. `COUNT(*)` then counts the
   rows inside each group, not in the table.
2. `channel` is legal in the `SELECT` because it is the grouping column — every row in a group has the same
   value for it, so there is one obvious answer.
3. In the second query, four different aggregates are computed per status in one pass over the table. This is
   what SQL is good at: you describe the summary, the engine makes one trip through the data.
4. `COUNT(DISTINCT customer_id)` counts unique customers per status, which is a different and usually more
   interesting number than the order count.
5. `ORDER BY orders DESC` sorts the groups by the aggregate. Sorting by something you computed is completely
   normal — the sort happens after the groups are built.

## 16. HAVING — Filtering Groups, Not Rows

`WHERE` filters rows **before** grouping. `HAVING` filters groups **after**. That is the whole difference, and
it explains the error you will hit sooner or later: `WHERE COUNT(*) > 5` is invalid, because at the moment
`WHERE` runs there are no groups yet and nothing has been counted.

So: conditions on raw columns go in `WHERE`, conditions on aggregates go in `HAVING`. A query often needs both,
and when it does, `WHERE` first is also the faster order — it throws rows away before the expensive grouping
starts.

In [ ]:
q("""
SELECT city,
       COUNT(*) AS customers
FROM customers
WHERE city IS NOT NULL
GROUP BY city
HAVING COUNT(*) >= 5
ORDER BY customers DESC, city
""")

**Step by step:**

1. `WHERE city IS NOT NULL` runs first and removes the five customers with no city, so they never form a group.
2. `GROUP BY city` makes one group per remaining city.
3. `HAVING COUNT(*) >= 5` inspects each finished group and keeps only the cities with five or more customers.
   Four cities survive.
4. Notice `COUNT(*)` is repeated in the `HAVING` rather than referring to the alias `customers`. SQLite allows
   the alias; PostgreSQL does not. Repeating the expression is the portable habit.
5. Swap the `WHERE` for `HAVING city IS NOT NULL` and you get the same answer here — but on a large table it is
   slower, because every `NULL` row is grouped first and thrown away afterwards.

## 17. CASE WHEN — If/Else Inside a Query

`CASE` is SQL's if/else. It walks the `WHEN` branches top to bottom, stops at the first one that is true, and
returns that `THEN` value. `ELSE` catches everything left over; without an `ELSE`, unmatched rows get `NULL`.

```sql
CASE WHEN condition THEN value
     WHEN condition THEN value
     ELSE value
END
```

Order matters, because the first match wins. `WHEN price >= 10000` written above `WHEN price >= 50000` would
swallow the expensive products before the second branch ever ran.

The trick worth learning early is putting `CASE` **inside an aggregate**: `SUM(CASE WHEN status = 'returned'
THEN 1 ELSE 0 END)` counts returns and normal orders in the same pass. That pattern — conditional aggregation —
is how you turn rows into a cross-tab, and the intermediate level leans on it hard.

In [ ]:
print(q("""
SELECT CASE WHEN price >= 50000 THEN 'premium'
            WHEN price >= 10000 THEN 'mid'
            ELSE 'budget'
       END      AS tier,
       COUNT(*) AS products,
       ROUND(AVG(price)) AS avg_price
FROM products
GROUP BY tier
ORDER BY avg_price DESC
""").to_string(index=False))

q("""
SELECT channel,
       COUNT(*)                                              AS orders,
       SUM(CASE WHEN status = 'cancelled' THEN 1 ELSE 0 END) AS cancelled,
       SUM(CASE WHEN status = 'returned'  THEN 1 ELSE 0 END) AS returned,
       ROUND(100.0 * SUM(CASE WHEN status IN ('cancelled', 'returned') THEN 1 ELSE 0 END)
             / COUNT(*), 1)                                  AS bad_pct
FROM orders
GROUP BY channel
ORDER BY bad_pct DESC
""")

**Step by step:**

1. The first `CASE` turns a number into a label. Because the branches are checked in order, a 62000 laptop
   matches `>= 50000` and never reaches the `mid` branch.
2. `GROUP BY tier` groups by the alias — the label the `CASE` produced. SQLite and MySQL allow that; in
   PostgreSQL you would repeat the whole `CASE` in the `GROUP BY` or wrap the query.
3. In the second query the `CASE` sits inside `SUM`. Each row contributes 1 or 0, so the sum is a count of the
   rows that matched — a count with a condition attached.
4. That is how you get two different counts over the same rows in one query. Doing it with `WHERE` would need
   two separate queries and something to stitch them together.
5. The `bad_pct` line divides one conditional count by the total. `100.0` keeps it out of integer division, and
   `ROUND(..., 1)` makes it readable.

## 18. COALESCE and NULLIF — Handling Missing Values

`COALESCE(a, b, c, ...)` returns the first argument that is not `NULL`. It is how you supply a default:
`COALESCE(city, 'unknown')`.

`NULLIF(a, b)` returns `NULL` when `a` equals `b`, and `a` otherwise. Its two everyday uses are turning a junk
placeholder into a proper `NULL` (`NULLIF(city, '')`) and guarding a division: `x / NULLIF(y, 0)` returns
`NULL` instead of raising a divide-by-zero, and `NULL` propagates harmlessly through the rest of the
calculation.

Wrapping everything in `COALESCE` out of habit is a mistake. `COALESCE(amount, 0)` in a `SUM` says "orders we
have no payment for were paid zero", which quietly turns missing data into a real number nobody can spot later.
Decide each time whether unknown really means zero.

In [ ]:
print(q("""
SELECT customer_id,
       name,
       COALESCE(city, 'unknown')  AS city,
       COALESCE(state, 'unknown') AS state
FROM customers
WHERE city IS NULL
ORDER BY customer_id
""").to_string(index=False))

q("""
SELECT name,
       price,
       cost,
       ROUND(price * 1.0 / NULLIF(cost, 0), 2) AS markup,
       ROUND(100.0 * (price - cost) / NULLIF(price, 0), 1) AS margin_pct
FROM products
ORDER BY markup DESC, name
LIMIT 5
""")

**Step by step:**

1. `COALESCE(city, 'unknown')` leaves a real city alone and replaces `NULL` with the word `unknown`. The stored
   data does not change — this is display only.
2. It takes as many arguments as you like and returns the first non-`NULL`: `COALESCE(nickname, name, 'guest')`
   reads like a fallback chain.
3. `NULLIF(cost, 0)` turns a zero cost into `NULL` before the division. No product has a zero cost here, but the
   guard costs nothing and stops the query breaking the day one does.
4. Without that guard, a zero would either raise an error or produce infinity depending on the database. With
   it you get `NULL`, which is honest.
5. `ROUND(..., 2)` and `ROUND(..., 1)` keep the two derived columns readable. `NULL` divided into anything stays
   `NULL`, so the guard survives the rounding.

## 19. INNER JOIN — Reading Two Tables at Once

`orders` stores a `customer_id`, not a customer name. To see the name you have to join.

```sql
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
```

`JOIN` (the full name is `INNER JOIN`; the word `INNER` is optional and almost never written) pairs each row on
the left with the matching rows on the right, using the condition after `ON`. **Rows with no match on either
side disappear.**

`o` and `c` are table aliases. Once a query touches more than one table, prefix every column with its alias —
`c.name`, `o.order_date`. It stops ambiguity errors when both tables have a column called `name`, and it makes
the query readable six months later.

More than two tables just means more `JOIN` lines. Each one joins to something already in the query.

In [ ]:
q("""
SELECT o.order_id,
       o.order_date,
       c.name  AS customer,
       c.city,
       p.name  AS product,
       i.quantity,
       i.unit_price
FROM orders o
JOIN customers   c ON c.customer_id = o.customer_id
JOIN order_items i ON i.order_id    = o.order_id
JOIN products    p ON p.product_id  = i.product_id
WHERE o.order_id <= 3
ORDER BY o.order_id, p.name
""")

**Step by step:**

1. `FROM orders o` starts with orders and calls the table `o` for the rest of the query.
2. The first `JOIN` attaches the customer. `ON c.customer_id = o.customer_id` is the condition — a foreign key
   meeting the primary key it points at, which is what the great majority of joins are.
3. The second `JOIN` attaches the order's items. This one is **one to many**: an order with three items becomes
   three rows. The order's date is repeated on each of them.
4. The third `JOIN` swaps each item's `product_id` for the product's details.
5. `WHERE o.order_id <= 3` keeps the output small enough to read. Notice the row count is bigger than three —
   that row multiplication is the single most important thing to understand about joins, and the intermediate
   level opens with what it does to your `SUM`.

## 20. LEFT JOIN — Keeping Rows That Have No Match

An `INNER JOIN` silently drops rows with no match. Nine of our customers have never ordered anything; join
`customers` to `orders` with a plain `JOIN` and those nine vanish, and nothing tells you.

`LEFT JOIN` keeps every row from the left table. Where there is no match on the right, the right-hand columns
come back as `NULL`.

That gives you the standard way to ask "which things have none of the other thing?" — `LEFT JOIN`, then
`WHERE right_table.some_column IS NULL`. It is worth learning as a shape, because it is everywhere: customers
who never ordered, products never sold, orders never paid.

One trap. A condition on the right-hand table belongs in the `ON`, not the `WHERE`. Putting it in the `WHERE`
tests it *after* the join has filled the missing side with `NULL`s, the test fails on those rows, and your
`LEFT JOIN` quietly becomes an inner one.

In [ ]:
print(q("""
SELECT c.customer_id,
       c.name,
       COUNT(o.order_id) AS orders
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
GROUP BY c.customer_id, c.name
HAVING COUNT(o.order_id) = 0
ORDER BY c.customer_id
""").to_string(index=False))

q("""
SELECT o.order_id,
       o.order_date,
       o.status,
       p.amount,
       p.method
FROM orders o
LEFT JOIN payments p ON p.order_id = o.order_id
WHERE p.payment_id IS NULL
ORDER BY o.order_id
LIMIT 6
""")

**Step by step:**

1. `FROM customers c LEFT JOIN orders o` keeps all 60 customers whether or not they ever ordered.
2. `COUNT(o.order_id)` counts the **order** column, not `*`. For a customer with no orders the joined row
   exists but `o.order_id` is `NULL`, so the count is 0. `COUNT(*)` would have counted that placeholder row and
   returned 1 for everybody — a classic and very quiet bug.
3. `HAVING COUNT(o.order_id) = 0` keeps only the customers who never ordered. Nine of them.
4. The second query is the same shape without the grouping: `LEFT JOIN payments`, then
   `WHERE p.payment_id IS NULL` keeps the orders that have no payment row. `payment_id` is the right column to
   test because it is the primary key and can never legitimately be `NULL`.
5. Move that condition into the `ON` (`ON p.order_id = o.order_id AND p.payment_id IS NULL`) and the answer
   changes completely. `ON` decides what counts as a match; `WHERE` filters what the join produced.

## 21. The Order a Query Actually Runs In

A query is written in one order and executed in another. Learning the execution order explains most of the
error messages you will hit:

| Step | Clause | What happens |
| --- | --- | --- |
| 1 | `FROM` / `JOIN` | assemble the rows |
| 2 | `WHERE` | drop rows |
| 3 | `GROUP BY` | form groups |
| 4 | `HAVING` | drop groups |
| 5 | `SELECT` | compute the output columns, apply aliases |
| 6 | `ORDER BY` | sort |
| 7 | `LIMIT` | cut |

Read that list and the rules stop being arbitrary. `WHERE` cannot use `COUNT(*)` because counting has not
happened yet. `HAVING` can, because it has. `ORDER BY` can use a `SELECT` alias because step 5 already ran;
`WHERE` cannot, because step 2 came first.

That last one has an asterisk. SQLite lets you use a `SELECT` alias in `WHERE` anyway, as a convenience.
PostgreSQL and SQL Server reject it. Write the full expression in `WHERE` and your query travels.

In [ ]:
q("""
SELECT channel,                    -- 5. build the output
       COUNT(*) AS orders
FROM orders                        -- 1. gather the rows
WHERE status <> 'cancelled'        -- 2. drop rows
GROUP BY channel                   -- 3. form groups
HAVING COUNT(*) > 30               -- 4. drop groups
ORDER BY orders DESC               -- 6. sort, and the alias exists by now
LIMIT 3                            -- 7. cut
""")

**Step by step:**

1. `FROM orders` puts 300 rows on the table.
2. `WHERE status <> 'cancelled'` removes 18 cancelled orders. This is row-level and happens before anything is
   counted.
3. `GROUP BY channel` turns the remaining rows into four groups.
4. `HAVING COUNT(*) > 30` drops the groups that are too small — group-level, and only possible after step 3.
5. `SELECT` computes `COUNT(*)` and names it `orders`; `ORDER BY orders DESC` then sorts using that name, and
   `LIMIT 3` cuts. Two comment styles work in SQL: `--` to the end of the line, and `/* ... */` for a block.

## 22. Creating and Changing Data — CREATE, INSERT, UPDATE, DELETE

Everything so far has read data. The four statements that change it:

- `CREATE TABLE name (column type constraints, ...)` makes a table.
- `INSERT INTO name (columns) VALUES (...), (...)` adds rows.
- `UPDATE name SET column = value WHERE ...` changes existing rows.
- `DELETE FROM name WHERE ...` removes rows.

**`UPDATE` and `DELETE` without a `WHERE` hit every row in the table.** There is no confirmation prompt and no
undo. The habit that saves you: write it as a `SELECT` first, look at the rows it returns, and only then swap
`SELECT ...` for `DELETE` or `UPDATE ... SET`.

Constraints are the database defending itself. `NOT NULL` rejects empty values, `PRIMARY KEY` rejects
duplicates, `REFERENCES` rejects a `customer_id` that does not exist, `CHECK` rejects anything you can express
as a condition. Every constraint you declare is a class of bug that can no longer reach your data.

In [ ]:
run("""
DROP TABLE IF EXISTS wishlist;

CREATE TABLE wishlist (
    wishlist_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL REFERENCES customers(customer_id),
    product_id  INTEGER NOT NULL REFERENCES products(product_id),
    added_on    TEXT    NOT NULL,
    priority    INTEGER NOT NULL DEFAULT 3 CHECK (priority BETWEEN 1 AND 5)
);

INSERT INTO wishlist (customer_id, product_id, added_on, priority) VALUES
    (1,  3, '2024-11-02', 1),
    (1, 27, '2024-11-05', 2),
    (4,  8, '2024-11-09', 3),
    (9,  3, '2024-12-01', 5);

UPDATE wishlist SET priority = 1 WHERE product_id = 3;
DELETE FROM wishlist WHERE customer_id = 9;
""")

try:
    run("INSERT INTO wishlist (customer_id, product_id, added_on) VALUES (999, 3, '2024-12-02')")
except Exception as e:
    print("rejected:", e)

q("SELECT * FROM wishlist ORDER BY wishlist_id")

**Step by step:**

1. `DROP TABLE IF EXISTS wishlist` makes the cell re-runnable. Without it, the second run would fail because the
   table already exists.
2. `INTEGER PRIMARY KEY` in SQLite auto-numbers itself, which is why the `INSERT` never supplies a
   `wishlist_id`. Other databases spell this `SERIAL` or `AUTO_INCREMENT` or `IDENTITY`.
3. `DEFAULT 3` fills `priority` in when the `INSERT` leaves it out; `CHECK (priority BETWEEN 1 AND 5)` refuses
   anything outside that range. `REFERENCES customers(customer_id)` refuses a customer who does not exist.
4. The `UPDATE` has a `WHERE`, so it touches two rows rather than all four. The `DELETE` removes the one row for
   customer 9. Both would have hit everything without their `WHERE`.
5. The `try` block deliberately breaks the foreign key with customer 999 and prints the rejection. That error is
   the constraint doing its job — the bad row never reaches the table.

## 23. SQL or pandas? — Choosing Where to Do the Work

You now know two ways to group and filter. Which one should do the job?

| Do it in SQL when | Do it in pandas when |
| --- | --- |
| the data is bigger than memory | the data already fits comfortably |
| you only need a summary of a big table | you need to plot it, or feed it to scikit-learn |
| joins and filters cut the rows down a lot | the reshaping is awkward to express in SQL |
| the logic belongs to everybody who queries this table | the work is one-off and exploratory |

The rule of thumb that survives contact with real work: **filter and aggregate in SQL, then bring the small
result into pandas.** Pulling a ten-million-row table into a DataFrame to run a `groupby` wastes network,
memory and time — the database was built to do exactly that and it is sitting next to the data.

The mirror image is also true. Do not build a monstrous 200-line query to do something pandas does in one
line, when the input is already small.

In [ ]:
in_sql = q("""
SELECT channel, COUNT(*) AS orders
FROM orders
WHERE status <> 'cancelled'
GROUP BY channel
ORDER BY orders DESC
""")

orders_df = q("SELECT channel, status FROM orders")
in_pandas = (orders_df[orders_df["status"] != "cancelled"]
             .groupby("channel").size()
             .reset_index(name="orders")
             .sort_values("orders", ascending=False)
             .reset_index(drop=True))

print("SQL")
print(in_sql.to_string(index=False))
print("\npandas")
print(in_pandas.to_string(index=False))
print("\nsame answer:", in_sql.equals(in_pandas))

**Step by step:**

1. The SQL version filters, groups, counts and sorts inside the database, and returns four rows.
2. The pandas version pulls 300 rows out first, then does the same four steps with `[]`, `groupby`, `size` and
   `sort_values`.
3. `reset_index(name="orders")` turns the grouped Series back into a DataFrame with a named column, so the two
   results are shaped the same way.
4. `in_sql.equals(in_pandas)` confirms they match. Both are correct — the difference is only where the work
   happened and how much data crossed the boundary.
5. At 300 rows this is a wash. At 300 million the SQL version still returns four rows and the pandas version
   never finishes, and that is the whole argument.

## 24. If You Move to PostgreSQL or MySQL

Everything in this guide is standard SQL apart from a handful of function names. The concepts — `SELECT`,
joins, `GROUP BY`, `NULL` semantics, execution order — are identical everywhere.

| Task | SQLite (here) | PostgreSQL | MySQL |
| --- | --- | --- | --- |
| Month bucket | `strftime('%Y-%m', d)` | `to_char(d, 'YYYY-MM')` | `DATE_FORMAT(d, '%Y-%m')` |
| Year part | `strftime('%Y', d)` | `EXTRACT(YEAR FROM d)` | `YEAR(d)` |
| Today | `date('now')` | `CURRENT_DATE` | `CURDATE()` |
| Add a week | `date(d, '+7 days')` | `d + INTERVAL '7 days'` | `DATE_ADD(d, INTERVAL 7 DAY)` |
| Join strings | `a \|\| b` | `a \|\| b` | `CONCAT(a, b)` |
| First non-null | `COALESCE(a, b)` | `COALESCE(a, b)` | `COALESCE(a, b)` |
| Auto id | `INTEGER PRIMARY KEY` | `GENERATED ALWAYS AS IDENTITY` | `AUTO_INCREMENT` |
| Limit rows | `LIMIT 10` | `LIMIT 10` | `LIMIT 10` |
| Describe a table | `PRAGMA table_info(t)` | `\d t` | `DESCRIBE t` |

Three behavioural differences worth carrying with you:

- **Types.** SQLite barely enforces them — put text in an integer column and it will often shrug. PostgreSQL
  will not. Code that works here can fail there, never the reverse, which is a good direction for a learning
  database to be wrong in.
- **`GROUP BY` strictness.** PostgreSQL rejects a `SELECT` column that is neither grouped nor aggregated.
  SQLite invents an answer. Write queries that would satisfy PostgreSQL.
- **Aliases in `WHERE`.** SQLite allows it, PostgreSQL does not. Repeat the expression.

In [ ]:
print("SQLite version:", q("SELECT sqlite_version() AS v")["v"][0])

q("""
SELECT strftime('%Y-%m', order_date) AS month,   -- to_char(order_date, 'YYYY-MM') in PostgreSQL
       COUNT(*)                      AS orders
FROM orders
WHERE order_date >= '2024-10-01'
GROUP BY month
ORDER BY month
""")

**Step by step:**

1. `sqlite_version()` reports the engine. Every database has an equivalent (`version()` in PostgreSQL and
   MySQL) and it is the first thing to check when a function you expect is missing.
2. The query itself is a monthly count — the single most common report shape there is.
3. Only one thing in it is SQLite-specific: `strftime`. The comment shows the PostgreSQL spelling; swap that one
   expression and the query runs unchanged.
4. `GROUP BY month` groups by the `SELECT` alias. PostgreSQL would want the full expression or a wrapping
   subquery, which is the kind of small edit that porting a query usually amounts to.
5. `WHERE order_date >= '2024-10-01'` is plain text comparison and portable as written, because the dates are
   stored in `YYYY-MM-DD`.

## 25. Common Pitfalls

The mistakes that cost beginners the most time, in rough order of how often they happen:

1. **`= NULL` instead of `IS NULL`.** Returns nothing, silently. See section 7.
2. **`<>` losing `NULL` rows.** `WHERE city <> 'Chennai'` drops the unknowns. Add `OR city IS NULL` when you
   want them.
3. **Integer division.** `(a - b) * 100 / a` truncates to a whole number. Use `100.0`.
4. **`COUNT(*)` after a `LEFT JOIN`.** Counts the `NULL` placeholder row too. Count a column from the right
   table instead.
5. **`LIMIT` without `ORDER BY`.** "Any 10 rows", not "the top 10", and the answer can change between runs.
6. **Double quotes around text.** `"Chennai"` means a column named Chennai. Text is single-quoted.
7. **`UPDATE` or `DELETE` with no `WHERE`.** The whole table. Write it as a `SELECT` first.
8. **`WHERE` on an aggregate.** Use `HAVING`; aggregates do not exist yet when `WHERE` runs.
9. **Selecting a column that is neither grouped nor aggregated.** SQLite guesses, PostgreSQL errors, you get a
   wrong number.
10. **`BETWEEN` on dates including the far end.** `BETWEEN '2024-01-01' AND '2024-02-01'` covers a day of
    February.

In [ ]:
print("integer vs real division")
print(q("SELECT 7 / 2 AS integer_division, 7.0 / 2 AS real_division").to_string(index=False))

print("\ncounting with NULLs")
print(q("SELECT COUNT(*) AS rows_, COUNT(city) AS with_city FROM customers").to_string(index=False))

print("\nthe <> leak")
print(q("""
SELECT (SELECT COUNT(*) FROM customers)                            AS everyone,
       (SELECT COUNT(*) FROM customers WHERE city =  'Chennai')    AS chennai,
       (SELECT COUNT(*) FROM customers WHERE city <> 'Chennai')    AS not_chennai,
       (SELECT COUNT(*) FROM customers WHERE city IS NULL)         AS unknown_city
""").to_string(index=False))

**Step by step:**

1. `7 / 2` is 3 and `7.0 / 2` is 3.5. One character between a right answer and a wrong one, with no warning.
2. `COUNT(*)` is 60 and `COUNT(city)` is 55. Both are correct answers to different questions; the bug is asking
   the wrong one.
3. The third query puts four counts side by side. `everyone` is 60, `chennai` is 12, `not_chennai` is 43 —
   and 12 + 43 is 55, not 60.
4. The missing five are `unknown_city`. `chennai + not_chennai + unknown_city` does add up, which is the point:
   with `NULL` around, "not X" is not the complement of "X".
5. Those `(SELECT ...)` brackets are **scalar subqueries** — a query returning exactly one value, used as if it
   were a value. The intermediate level goes into them properly.

## Cheat Sheet

| Task | SQL |
| --- | --- |
| All columns | `SELECT * FROM products` |
| Named columns, renamed | `SELECT name AS product, price FROM products` |
| Filter rows | `WHERE price > 50000` |
| Combine conditions | `WHERE (a = 1 OR a = 2) AND b < 10` |
| Range | `WHERE price BETWEEN 1000 AND 5000` |
| List of values | `WHERE category_id IN (1, 2, 3)` |
| Text pattern | `WHERE name LIKE '%Laptop%'` |
| Missing value | `WHERE city IS NULL` / `IS NOT NULL` |
| Sort | `ORDER BY price DESC, name` |
| First n rows | `LIMIT 10` / `LIMIT 10 OFFSET 20` |
| Unique values | `SELECT DISTINCT city FROM customers` |
| Count distinct | `COUNT(DISTINCT city)` |
| Computed column | `SELECT price - cost AS margin` |
| Safe percentage | `ROUND(100.0 * a / b, 1)` |
| Text pieces | `UPPER(s)`, `LENGTH(s)`, `TRIM(s)`, `SUBSTR(s, 1, 3)`, `a \|\| b` |
| Month bucket | `strftime('%Y-%m', order_date)` |
| Date maths | `date(order_date, '+7 days')`, `julianday(a) - julianday(b)` |
| Aggregates | `COUNT(*)`, `SUM(x)`, `AVG(x)`, `MIN(x)`, `MAX(x)` |
| Rows vs non-null | `COUNT(*)` vs `COUNT(column)` |
| One row per group | `GROUP BY channel` |
| Filter groups | `HAVING COUNT(*) >= 5` |
| If/else | `CASE WHEN x THEN a ELSE b END` |
| Count with a condition | `SUM(CASE WHEN status = 'returned' THEN 1 ELSE 0 END)` |
| Default for missing | `COALESCE(city, 'unknown')` |
| Guard a division | `x / NULLIF(y, 0)` |
| Join two tables | `FROM orders o JOIN customers c ON c.customer_id = o.customer_id` |
| Keep unmatched rows | `LEFT JOIN payments p ON p.order_id = o.order_id` |
| Rows with no match | `LEFT JOIN ... WHERE p.payment_id IS NULL` |
| New table | `CREATE TABLE t (id INTEGER PRIMARY KEY, name TEXT NOT NULL)` |
| Add rows | `INSERT INTO t (name) VALUES ('a'), ('b')` |
| Change rows | `UPDATE t SET name = 'c' WHERE id = 1` |
| Remove rows | `DELETE FROM t WHERE id = 1` |
| List the tables | `SELECT name FROM sqlite_master WHERE type = 'table'` |
| Describe a table | `PRAGMA table_info(customers)` |

## Suggested Learning Path

1. Run section 1 and make sure the row counts print. Everything else depends on it.
2. Read sections 2 to 6 in one sitting — schema, `SELECT`, `WHERE`, and the shorthand. Do exercises 3 to 6.
3. Slow down on section 7. `NULL` causes more wrong answers than any other single thing in SQL.
4. Sections 8 to 13 are the row-level toolkit: sorting, slicing, arithmetic, text, dates. Do exercises 8 to 13.
5. Sections 14 to 16 are the shift from rows to summaries. Re-read `WHERE` versus `HAVING` until it is obvious.
6. Sections 17 and 18 give you conditional logic. `SUM(CASE WHEN ...)` is worth memorising as a shape.
7. Sections 19 and 20 are joins. Draw the two tables on paper and mark which rows survive each kind.
8. Read section 21 twice. The execution order explains the errors from all of the above.
9. Section 22 changes data. Practise the "write it as a `SELECT` first" habit before you ever run a `DELETE`.
10. Skim 23 and 24, then work through section 25 and check you can explain every one of the ten pitfalls.
11. Open `sql-basics-exercises.ipynb` and do all 24. The two mini projects at the end are the real test.

## Where to Go Next

- **`sql-intermediate`** is the natural next step: every join type, subqueries, CTEs, and window functions —
  the tools that turn "I can read one table" into "I can answer a business question".
- Install PostgreSQL, or run it in Docker, and re-point the same queries at it. Porting your own work is the
  fastest way to learn which parts were dialect and which were SQL.
- Read the schema of a database you already use at work. Finding the primary and foreign keys of a real system
  teaches more about design than any tutorial.
- Keep [SQLite's documentation](https://www.sqlite.org/lang.html) open. It is short, precise, and one of the
  better reference manuals in software.

## Practice Next

Open `sql-basics-exercises.ipynb`. It follows this guide section by section, every exercise names the section it
practises, and the checks tell you when you have it right. Do not read ahead to the intermediate level until the
two mini projects at the end run clean.